# Model Analysis
Audit the purged temporal split, target distributions, final-test metrics, discrimination, calibration, and segment implications without retraining inside the notebook.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import precision_recall_curve, roc_curve
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
snapshots = pd.read_csv(ROOT/'data/interim/user_snapshots.csv', parse_dates=['snapshot_date','ltv_label_mature_date','churn_label_mature_date'])
ltv = pd.read_csv(ROOT/'data/processed/ltv_predictions.csv')
churn = pd.read_csv(ROOT/'data/processed/churn_predictions.csv')
ltv_metrics = json.loads((ROOT/'reports/metrics/ltv_metrics.json').read_text())
churn_metrics = json.loads((ROOT/'reports/metrics/churn_metrics.json').read_text())
ltv_metrics['temporal_design']

## Purged time-split audit
The 90-day target maturity date for every training row must be on or before the validation snapshot; the validation label must mature by the test snapshot.

In [ ]:
split_counts = snapshots.groupby(['snapshot_date','ltv_split']).size().unstack(fill_value=0)
display(split_counts)
validation_date = snapshots.loc[snapshots.ltv_split.eq('validation'),'snapshot_date'].min()
test_date = snapshots.loc[snapshots.ltv_split.eq('test'),'snapshot_date'].min()
assert snapshots.loc[snapshots.ltv_split.eq('train'),'ltv_label_mature_date'].le(validation_date).all()
assert snapshots.loc[snapshots.ltv_split.eq('validation'),'ltv_label_mature_date'].le(test_date).all()
plot_counts = split_counts.copy(); plot_counts.index = plot_counts.index.strftime('%Y-%m-%d')
ax = plot_counts.plot.bar(stacked=True, figsize=(12,4), title='LTV rows by snapshot and split')
ax.set_ylabel('Rows'); ax.grid(axis='y', alpha=.2); plt.tight_layout()

## Target distributions

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))
sns.histplot(snapshots.loc[snapshots.ltv_split.isin(['train','validation']),'future_90d_revenue'], bins=40, ax=axes[0])
axes[0].set_title('Development 90-day LTV target')
snapshots.groupby('churn_split').churn_30d.mean().plot.bar(ax=axes[1], title='Churn rate by temporal split')
axes[1].set_ylabel('Churn rate'); axes[1].set_ylim(0,1)
plt.tight_layout()

## Final-test discrimination and calibration

In [ ]:
fpr, tpr, _ = roc_curve(churn.churn_30d, churn.churn_probability)
precision, recall, _ = precision_recall_curve(churn.churn_30d, churn.churn_probability)
fig, axes = plt.subplots(1,2,figsize=(11,4))
axes[0].plot(fpr,tpr); axes[0].plot([0,1],[0,1],'--',color='gray'); axes[0].set(title='Final-test ROC',xlabel='FPR',ylabel='TPR')
axes[1].plot(recall,precision); axes[1].set(title='Final-test precision-recall',xlabel='Recall',ylabel='Precision')
for ax in axes: ax.grid(alpha=.2)
plt.tight_layout()
display(pd.DataFrame(churn_metrics).T)

## LTV error analysis

In [ ]:
ltv['absolute_error'] = (ltv.future_90d_revenue - ltv.predicted_ltv).abs()
display(ltv.nlargest(15,'absolute_error')[['user_id','future_90d_revenue','predicted_ltv','absolute_error']])
fig, ax = plt.subplots(figsize=(6,5))
ax.scatter(ltv.future_90d_revenue, ltv.predicted_ltv, alpha=.25, s=12)
limit = max(ltv.future_90d_revenue.max(), ltv.predicted_ltv.max()); ax.plot([0,limit],[0,limit],'--',color='gray')
ax.set(xlabel='Actual 90-day revenue',ylabel='Predicted LTV',title='Final-test LTV: actual vs predicted'); ax.grid(alpha=.2)
plt.tight_layout()

## Interpretation
Low LTV R² is reported rather than hidden: the synthetic features explain little of future spend. Use the error table to identify heavy-tail failures. Churn thresholds are selected on validation, while test metrics are final evidence only. Segment labels support prioritization hypotheses, not automated customer treatment.